# 01 · Data and Split

This notebook acquires (or verifies) the PhiUSIIL snapshot, shows with aggregates why the
benchmark uses only host features with `www.` removed, and regenerates the frozen
domain-disjoint 24/12 split with the project's own preparation script. Every regenerated file
is compared with the versioned one, and any difference fails loudly. Nothing here is tuned: the
seed, sample sizes, candidate features and selection rule are fixed in
`src/data/benchmark_split.py` and `docs/benchmark-protocol.md`. All numbers in this notebook
come from the classical data pipeline; no model and no quantum computation is involved.

**Inputs**
- `data/raw/` (downloaded from UCI by `scripts/download_dataset.py` if absent, about 15 MB)
- `data/README.md` (documented checksums and counts)
- `docs/benchmark-protocol.md` (documented shortcut audit and exclusion counts)
- `configs/benchmark-split.json` (versioned split manifest, used as the reference)

**Outputs**
- `data/raw/PhiUSIIL_Phishing_URL_Dataset.csv`, `data/raw/phiusiil.zip`, `data/raw/acquisition.json` (only if they were absent)
- `configs/benchmark-split.json` (regenerated; must be identical to the versioned file)
- `data/processed/benchmark-sample.csv` (local, Git-ignored rows with hosts and features)

In [1]:
import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys
from pathlib import Path

ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "src").is_dir() and (p / "configs").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Start Jupyter from the repository root or notebooks/: src/ and configs/ not found above the current directory"
    )
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd

from src.data import benchmark_split as split
from src.features.host_features import CANDIDATE_FEATURES, host_features, normalize_host


def load_script(name):
    # Import a file from scripts/ as a module, without running its __main__ block.
    spec = importlib.util.spec_from_file_location(name, ROOT / "scripts" / f"{name}.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def rel(path):
    return Path(path).resolve().relative_to(ROOT.resolve()).as_posix()


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


print("Repository root located; working directory set to it.")
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

Repository root located; working directory set to it.
Python 3.12.10 | pandas 3.1.0 | numpy 2.5.4


## 1. Acquire or verify the snapshot

`acquire()` from `scripts/download_dataset.py` reuses an existing `data/raw/` only after
re-hashing both files and re-running the full structural CSV inspection against
`acquisition.json`. If `data/raw/` is absent it downloads the original UCI ZIP, validates it in
a staging directory and publishes the snapshot at once. An inconsistent existing snapshot
raises an error instead of being replaced.

In [2]:
download = load_script("download_dataset")
manifest_raw, reused = download.acquire()
print(
    "Verified existing snapshot; no download performed."
    if reused
    else "Downloaded and validated a fresh snapshot."
)

raw_dir = download.DEFAULT_DESTINATION
files = pd.DataFrame(
    [
        {
            "file": rel(raw_dir / record["filename"]),
            "bytes": record["bytes"],
            "sha256": record["sha256"],
        }
        for record in manifest_raw["files"].values()
    ]
)
print(f"Acquired at (UTC): {manifest_raw['acquired_at_utc']}")
print(f"Source version: {manifest_raw['source_version']} ({manifest_raw['checksum_note']})")
files

Downloaded and validated a fresh snapshot.
Acquired at (UTC): 2026-10-25T10:40:53.714035+00:00
Source version: unavailable (Locally computed identity hashes; no official reference hash provided.)


,file,bytes,sha256
0,data/raw/phiusiil.zip,15376007,0a639fd03aea6308c5b1c10c92aa23c2ce1505447a9137...
1,data/raw/PhiUSIIL_Phishing_URL_Dataset.csv,56854345,a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7...


**Replication check.** The recorded sizes and SHA-256 values must equal the ones documented
in `data/README.md`, and the files on disk must hash to them. A future UCI download could
have different hashes; in that case this check fails, which means the documented snapshot is
not the one being used.

In [3]:
readme = (ROOT / "data" / "README.md").read_text(encoding="utf-8")
documented = {}
for key, label in (("archive", "Archive"), ("csv", "CSV")):
    match = re.search(rf"- {label}: ([\d,]+) bytes; SHA-256 `([0-9a-f]{{64}})`", readme)
    assert match, f"Documented {label} checksum not found in data/README.md"
    documented[key] = {"bytes": int(match.group(1).replace(",", "")), "sha256": match.group(2)}

for key, record in manifest_raw["files"].items():
    path = raw_dir / record["filename"]
    assert record["bytes"] == documented[key]["bytes"], f"{key}: size differs from data/README.md"
    assert record["sha256"] == documented[key]["sha256"], (
        f"{key}: SHA-256 differs from data/README.md"
    )
    assert path.stat().st_size == record["bytes"], f"{key}: size on disk differs"
    assert sha256(path) == record["sha256"], f"{key}: SHA-256 on disk differs"
    print(f"OK: {rel(path)} has {record['bytes']:,} bytes and the documented SHA-256.")

inspection = manifest_raw["inspection"]
assert inspection["counts_match_published"] and inspection["warnings"] == []
assert sum(inspection["empty_cells_by_column"].values()) == 0
print("OK: counts match the UCI-published counts; no warnings; no empty cells.")

OK: data/raw/phiusiil.zip has 15,376,007 bytes and the documented SHA-256.
OK: data/raw/PhiUSIIL_Phishing_URL_Dataset.csv has 56,854,345 bytes and the documented SHA-256.
OK: counts match the UCI-published counts; no warnings; no empty cells.


## 2. Dataset overview

Original labels are `1 = legitimate` and `0 = phishing`. The modeling target is
`is_phishing = 1 - label`, so phishing is the positive class.

The dataset is Prasad, A. & Chandra, S. (2024), *PhiUSIIL Phishing URL (Website)*, UCI Machine
Learning Repository, distributed under CC BY 4.0. Derived data must keep the attribution, the
license link and an indication of modifications. The repository's MIT license does not
replace the dataset license. Do not visit the URLs in this dataset.

In [4]:
CSV_PATH = raw_dir / download.CSV_NAME
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig", keep_default_na=False)
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")
print(f"Attribution: {manifest_raw['attribution']}")
print(f"License: {manifest_raw['license']['name']} ({manifest_raw['license']['url']})")
print(f"Source page: {manifest_raw['source_page']}")

balance = raw["label"].value_counts().sort_index().rename_axis("label").rename("rows").to_frame()
balance["meaning"] = balance.index.map({0: "phishing", 1: "legitimate"})
balance["is_phishing"] = 1 - balance.index
balance["share"] = (balance["rows"] / balance["rows"].sum()).round(4)
balance

Rows: 235,795; columns: 56
Attribution: Prasad, A. & Chandra, S. (2024). PhiUSIIL. UCI Machine Learning Repository.
License: CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/)
Source page: https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset


,rows,meaning,is_phishing,share
label,,,,
0,100945,phishing,1,0.4281
1,134850,legitimate,0,0.5719


In [5]:
# Replication check: counts against the downloader's published constants and data/README.md.
expected = download.EXPECTED_COUNTS
assert len(raw) == expected["rows"] == inspection["rows"] == 235_795
assert int((raw["label"] == 0).sum()) == expected["labels"]["0"] == 100_945
assert int((raw["label"] == 1).sum()) == expected["labels"]["1"] == 134_850
assert raw.shape[1] == inspection["column_count"] == 56
assert "235,795 data rows and 56 columns" in readme
print("OK: 235,795 rows (100,945 phishing, 134,850 legitimate) and 56 columns, as documented.")

OK: 235,795 rows (100,945 phishing, 134,850 legitimate) and 56 columns, as documented.


## 3. Shortcut audit

PhiUSIIL was collected differently for each class, and some fields separate the classes
without describing phishing. The table below recomputes, per class, the share of rows showing
each pattern documented in `docs/benchmark-protocol.md`. "Path or query" means any character
after the host (a `/` or `?`), so a bare trailing slash counts. Only aggregates are shown.

In [6]:
is_phishing = raw["label"] == 0
after_scheme = raw["URL"].str.replace(r"^[A-Za-z][A-Za-z0-9+.-]*://", "", regex=True)
patterns = {
    "URL has a path or query": after_scheme.str.contains(r"[/?]", regex=True),
    "Host starts with `www.`": raw["Domain"].str.lower().str.startswith("www."),
    "`IsHTTPS` equals 1": raw["IsHTTPS"] == 1,
    "`URLSimilarityIndex` equals 100": raw["URLSimilarityIndex"] == 100,
}
audit = pd.DataFrame(
    {
        "legitimate_pct": {
            name: 100 * flag[~is_phishing].mean() for name, flag in patterns.items()
        },
        "phishing_pct": {name: 100 * flag[is_phishing].mean() for name, flag in patterns.items()},
    }
)
bare_homepage = raw["URL"].str.fullmatch(r"https://www\.[^/?#]+/?")
print(
    f"Legitimate URLs that are a bare https://www.<host> homepage: {100 * bare_homepage[~is_phishing].mean():.2f}%"
)
print(
    f"Phishing URLs that are a bare https://www.<host> homepage:   {100 * bare_homepage[is_phishing].mean():.2f}%"
)
audit.round(2)

Legitimate URLs that are a bare https://www.<host> homepage: 100.00%
Phishing URLs that are a bare https://www.<host> homepage:   1.03%


,legitimate_pct,phishing_pct
URL has a path or query,0.0,59.87
Host starts with `www.`,100.0,41.36
`IsHTTPS` equals 1,100.0,49.22
`URLSimilarityIndex` equals 100,100.0,0.78


In [7]:
# Replication check: every recomputed share must round to the value printed in the protocol table.
protocol = (ROOT / "docs" / "benchmark-protocol.md").read_text(encoding="utf-8")
table = protocol.split("| Pattern in the source data |", 1)[1].split("\n\n", 1)[0]
documented_rows = {}
for line in table.strip().splitlines()[2:]:
    name, legit, phish = [cell.strip() for cell in line.strip().strip("|").split("|")]
    documented_rows[name] = (legit, phish)
assert set(documented_rows) == set(audit.index), sorted(documented_rows)


def rounds_to(value, text):
    shown = text.rstrip("%")
    decimals = len(shown.split(".")[1]) if "." in shown else 0
    return round(value, decimals) == float(shown)


for name, (legit, phish) in documented_rows.items():
    assert rounds_to(audit.loc[name, "legitimate_pct"], legit), (
        name,
        audit.loc[name, "legitimate_pct"],
        legit,
    )
    assert rounds_to(audit.loc[name, "phishing_pct"], phish), (
        name,
        audit.loc[name, "phishing_pct"],
        phish,
    )
    print(f"OK: {name}: legitimate {legit}, phishing {phish}, as in the protocol.")
assert bare_homepage[~is_phishing].all()
print("OK: every legitimate URL is a bare https://www.<host> homepage.")

OK: URL has a path or query: legitimate 0%, phishing 60%, as in the protocol.
OK: Host starts with `www.`: legitimate 100%, phishing 41%, as in the protocol.
OK: `IsHTTPS` equals 1: legitimate 100%, phishing 49%, as in the protocol.
OK: `URLSimilarityIndex` equals 100: legitimate 100%, phishing 0.8%, as in the protocol.
OK: every legitimate URL is a bare https://www.<host> homepage.


Every legitimate URL is a bare `https://www.<host>` homepage, while most phishing URLs are
not. A classifier given the scheme, path, query or `www.` prefix would learn how the dataset
was collected, not what phishing looks like. Page features (for example `LineOfCode`) would
need a page fetch, and `URLSimilarityIndex`, `TLDLegitimateProb` and `URLCharProb` depend on
reference statistics that cannot be rebuilt here. The benchmark therefore computes its
candidate features from the host alone, after `normalize_host` removes one leading `www.`.
After that normalization the `www.` prefix no longer separates the classes:

In [8]:
hosts = raw["Domain"].astype(str).map(normalize_host)
still_www = hosts.str.startswith("www.")
print(
    f"Hosts still starting with www. after normalization: legitimate {100 * still_www[~is_phishing].mean():.3f}%, "
    f"phishing {100 * still_www[is_phishing].mean():.3f}%"
)

# Made-up example hosts (not from the dataset), showing normalization and the eight candidates.
examples = ["WWW.Example.com.", "www.login-example-bank.web.app:443", "192.0.2.10"]
pd.DataFrame(
    [{"input": h, "host": normalize_host(h), **host_features(normalize_host(h))} for h in examples]
).set_index("input")

Hosts still starting with www. after normalization: legitimate 0.000%, phishing 0.001%


,host,host_length,host_digit_ratio,host_hyphen_count,host_label_count,host_entropy,longest_label_length,tld_length,host_is_ip
input,,,,,,,,,
WWW.Example.com.,example.com,11.0,0.0,0.0,2.0,3.095795,7.0,3.0,0.0
www.login-example-bank.web.app:443,login-example-bank.web.app,26.0,0.0,2.0,3.0,3.767183,18.0,3.0,0.0
192.0.2.10,192.0.2.10,10.0,0.7,0.0,4.0,2.246439,3.0,0.0,1.0


## 4. Regenerate the frozen split

`scripts/prepare_benchmark.py` checks the CSV hash against `acquisition.json`, deduplicates
URLs, groups hosts by registrable domain (bundled Public Suffix List, private suffixes
included, IP hosts by address), drops unusable hosts and label-conflicting groups, draws the
test rows first, selects four features on the training-side pool, then draws the training
rows. It overwrites `configs/benchmark-split.json` and `data/processed/benchmark-sample.csv`,
so the reference manifest is read first and compared afterwards. The reference is Git's
committed version (`HEAD:configs/benchmark-split.json`) when the folder is a Git checkout,
otherwise the file as it was before this run (for example in a ZIP download).

The comparison is on the parsed JSON, not on raw bytes: `write_text` writes CRLF line endings
on Windows and LF elsewhere, and the manifest's `environment` block records the Python and
package versions of whoever ran it. Every key except `environment` must be equal; an
environment difference is only reported. The SHA-256 shown is of the LF-normalized bytes, so
it matches Git's blob on every platform.

In [9]:
MANIFEST_PATH = ROOT / "configs" / "benchmark-split.json"
SAMPLE_PATH = ROOT / "data" / "processed" / "benchmark-sample.csv"


def lf(data):
    return data.replace(b"\r\n", b"\n")


def committed_manifest():
    # Git's committed manifest when this is a Git checkout; None otherwise.
    if not (ROOT / ".git").exists():
        return None
    try:
        result = subprocess.run(
            ["git", "show", "HEAD:configs/benchmark-split.json"],
            cwd=ROOT,
            capture_output=True,
            check=True,
        )
    except (OSError, subprocess.CalledProcessError):
        return None
    return result.stdout


pre_run_manifest_bytes = MANIFEST_PATH.read_bytes()
reference_bytes = committed_manifest()
reference_source = "git HEAD"
if reference_bytes is None:
    reference_bytes, reference_source = (
        pre_run_manifest_bytes,
        "file before this run (no Git checkout)",
    )
previous_sample_bytes = SAMPLE_PATH.read_bytes() if SAMPLE_PATH.exists() else None
print(
    f"Reference {rel(MANIFEST_PATH)} from {reference_source}: "
    f"SHA-256 of LF-normalized bytes {hashlib.sha256(lf(reference_bytes)).hexdigest()[:16]}..."
)
print(
    f"Existing {rel(SAMPLE_PATH)}: {'present' if previous_sample_bytes is not None else 'absent'}"
)

prepare = load_script("prepare_benchmark")
return_code = prepare.main()
assert return_code == 0, f"prepare_benchmark.py returned {return_code}"

Reference configs/benchmark-split.json from file before this run (no Git checkout): SHA-256 of LF-normalized bytes b0fef0204be13483...
Existing data/processed/benchmark-sample.csv: absent


Features: host_entropy, host_label_count, host_digit_ratio, host_hyphen_count
Train: 24 rows; test: 12 rows.
Exclusions: {'duplicate_url': 425, 'unusable_host': 2, 'label_conflicting_group_rows': 606, 'label_conflicting_groups': 106}
Manifest: configs\benchmark-split.json


In [10]:
# Replication check: the regenerated manifest equals the reference in every key except "environment".
regenerated_bytes = MANIFEST_PATH.read_bytes()
manifest = json.loads(regenerated_bytes)
reference = json.loads(reference_bytes)
content_keys = sorted((set(reference) | set(manifest)) - {"environment"})
differing = [key for key in content_keys if reference.get(key) != manifest.get(key)]
if differing:
    MANIFEST_PATH.write_bytes(pre_run_manifest_bytes)  # restore the file this run overwrote
    raise AssertionError(
        f"Regenerated manifest differs from the reference ({reference_source}) in: {differing}. "
        f"{rel(MANIFEST_PATH)} was restored to its pre-run content; "
        "run `git checkout -- configs/benchmark-split.json` if that was not the committed version."
    )
print(
    f"OK: regenerated {rel(MANIFEST_PATH)} equals the reference ({reference_source}) in {content_keys}."
)
if lf(regenerated_bytes) == lf(reference_bytes):
    digest = hashlib.sha256(lf(reference_bytes)).hexdigest()[:16]
    print(f"OK: LF-normalized bytes are identical too (SHA-256 {digest}...).")

pinned_env, run_env = reference.get("environment", {}), manifest.get("environment", {})
if run_env != pinned_env:
    changed = {
        k: {"pinned": pinned_env.get(k), "this_run": run_env.get(k)}
        for k in sorted(set(pinned_env) | set(run_env))
        if pinned_env.get(k) != run_env.get(k)
    }
    print("WARNING: environment differs from the pinned one:", changed)
    print("         The split content above is identical, so this does not affect replication.")
else:
    print("OK: environment matches the pinned one:", run_env)

if previous_sample_bytes is not None:
    assert lf(SAMPLE_PATH.read_bytes()) == lf(previous_sample_bytes), "benchmark-sample.csv changed"
    print(f"OK: regenerated {rel(SAMPLE_PATH)} is identical to the previous local copy.")

OK: regenerated configs/benchmark-split.json equals the reference (file before this run (no Git checkout)) in ['counts', 'dataset', 'feature_selection', 'features', 'rules', 'schema_version', 'test', 'train'].
OK: LF-normalized bytes are identical too (SHA-256 b0fef0204be13483...).
OK: environment matches the pinned one: {'python': '3.12.10', 'numpy': '2.5.4', 'pandas': '3.1.0', 'scikit-learn': '1.9.1', 'scipy': '1.18.1', 'tldextract': '5.4.0', 'public_suffix_list': 'tldextract bundled snapshot; private suffixes included'}


In [11]:
# Replication check: manifest content against the protocol and the acquisition record.
rules, counts, selection = manifest["rules"], manifest["counts"], manifest["feature_selection"]
FROZEN_FEATURES = ["host_entropy", "host_label_count", "host_digit_ratio", "host_hyphen_count"]

assert manifest["dataset"] == {
    "csv_sha256": manifest_raw["files"]["csv"]["sha256"],
    "csv_rows": 235_795,
}
assert rules["seed"] == split.SEED == 42
assert (rules["train_per_class"], rules["test_per_class"]) == (12, 6)
assert rules["order"].startswith("test drawn first")
assert counts["exclusions"] == {
    "duplicate_url": 425,
    "unusable_host": 2,
    "label_conflicting_group_rows": 606,
    "label_conflicting_groups": 106,
}
assert counts["prepared_rows"] == 235_795 - 425 - 2 - 606
assert manifest["features"] == selection["selected"] == FROZEN_FEATURES
assert selection["candidates"] == list(CANDIDATE_FEATURES)
assert selection["selection_rows"] == 2 * split.SELECTION_SAMPLE_PER_CLASS == 20_000
assert split.MAX_ABS_SPEARMAN == 0.7 and "0.7" in selection["method"]
assert selection["skipped_as_redundant"] == {"host_length": "|Spearman| >= 0.7 with host_entropy"}
for part, size in (("train", 24), ("test", 12)):
    rows = manifest[part]
    assert len(rows) == size and [r["position"] for r in rows] == list(range(size))
    assert sum(r["is_phishing"] for r in rows) == size // 2
train_groups = {r["group_hash"] for r in manifest["train"]}
test_groups = {r["group_hash"] for r in manifest["test"]}
assert len(train_groups) == 24 and len(test_groups) == 12 and not train_groups & test_groups
print("OK: seed 42; 24 train (12/12) and 12 test (6/6); 36 distinct groups, none shared.")
print(
    "OK: exclusions 425 duplicate URLs, 2 unusable hosts, 106 conflicting groups (606 rows), as in the protocol."
)
print("OK: features", FROZEN_FEATURES, "frozen in this order; host_length skipped as redundant.")

pd.DataFrame(
    {
        "partition": ["train", "test"],
        "rows": [len(manifest["train"]), len(manifest["test"])],
        "phishing": [sum(r["is_phishing"] for r in manifest[p]) for p in ("train", "test")],
        "legitimate": [sum(1 - r["is_phishing"] for r in manifest[p]) for p in ("train", "test")],
    }
).set_index("partition")

OK: seed 42; 24 train (12/12) and 12 test (6/6); 36 distinct groups, none shared.
OK: exclusions 425 duplicate URLs, 2 unusable hosts, 106 conflicting groups (606 rows), as in the protocol.
OK: features ['host_entropy', 'host_label_count', 'host_digit_ratio', 'host_hyphen_count'] frozen in this order; host_length skipped as redundant.


,rows,phishing,legitimate
partition,,,
train,24,12,12
test,12,6,6


**Sample against manifest.** The local sample holds the hosts, so the checks below recompute
each row's group hash, label and features from it and from the raw CSV, and display only
`row_index` and `group_hash`.

In [12]:
sample = pd.read_csv(SAMPLE_PATH, keep_default_na=False)
assert list(sample.columns) == ["row_index", "host", "is_phishing", *FROZEN_FEATURES, "partition"]

for part in ("train", "test"):
    rows = sample[sample["partition"] == part].reset_index(drop=True)
    recorded = pd.DataFrame(manifest[part])
    assert rows["row_index"].tolist() == recorded["row_index"].tolist(), (
        f"{part}: row order differs"
    )
    assert rows["is_phishing"].tolist() == recorded["is_phishing"].tolist(), (
        f"{part}: labels differ"
    )
    hashes = rows["host"].map(lambda h: split.group_hash(split.group_key(h)))
    assert hashes.tolist() == recorded["group_hash"].tolist(), f"{part}: group hashes differ"
    source = raw.loc[rows["row_index"]]
    assert (
        source["Domain"].astype(str).map(normalize_host).to_numpy() == rows["host"].to_numpy()
    ).all()
    assert ((1 - source["label"]).to_numpy() == rows["is_phishing"].to_numpy()).all()
    recomputed = pd.DataFrame([host_features(h) for h in rows["host"]])[FROZEN_FEATURES]
    assert np.allclose(recomputed.to_numpy(), rows[FROZEN_FEATURES].to_numpy(), rtol=0, atol=1e-9)
    print(
        f"OK: {part}: {len(rows)} rows match the manifest (order, labels, group hashes), the raw CSV and host_features."
    )

assert sample["row_index"].is_unique and len(sample) == 36
print(f"OK: {rel(SAMPLE_PATH)} has 36 distinct rows (24 train + 12 test).")
pd.DataFrame(manifest["test"]).set_index("position")

OK: train: 24 rows match the manifest (order, labels, group hashes), the raw CSV and host_features.
OK: test: 12 rows match the manifest (order, labels, group hashes), the raw CSV and host_features.
OK: data/processed/benchmark-sample.csv has 36 distinct rows (24 train + 12 test).


,row_index,group_hash,is_phishing
position,,,
0,15517,fd393998db75545c,0
1,174792,b2d1a9ba5000a09a,0
2,217648,84d275114cab3730,0
3,161204,c660c6c6a7b6fffe,0
4,94665,010ccedeafebd986,0
5,18899,9a021f91155117e4,0
6,219334,55624b9e9738e434,1
7,41538,e51ce3301401094b,1
8,121258,62c0d81c126f3930,1


## 5. Order of operations

`split.build` runs the steps below in this order with one random generator seeded with 42.
The cell below mirrors `build()` step by step with the same functions, to show that the test
rows are drawn before anything else, that feature selection sees only the pool without test
groups, and that the training rows come from that pool. It then calls `split.build` itself and
asserts that the step-by-step replay gives exactly the same result, so the order shown here is
the order the project's function uses.

In [13]:
prepared = split.prepare(raw)
rng = np.random.default_rng(split.SEED)
test = split._one_row_per_group(prepared.rows, split.TEST_PER_CLASS, rng)  # 1. test first
pool = prepared.rows[
    ~prepared.rows["group"].isin(set(test["group"]))
]  # 2. pool without test groups
replayed_selection = split.select_features(pool, rng)  # 3. features on the pool
train = split._one_row_per_group(pool, split.TRAIN_PER_CLASS, rng)  # 4. training from the pool

assert prepared.exclusions == counts["exclusions"] and len(prepared.rows) == counts["prepared_rows"]
assert len(pool) == counts["training_pool_rows"]
assert test["row_index"].tolist() == [r["row_index"] for r in manifest["test"]]
assert train["row_index"].tolist() == [r["row_index"] for r in manifest["train"]]
assert {k: selection[k] for k in replayed_selection} == replayed_selection
assert not set(pool["group"]) & set(test["group"])

built = split.build(raw)
assert built["test"]["row_index"].tolist() == test["row_index"].tolist()
assert built["train"]["row_index"].tolist() == train["row_index"].tolist()
assert built["selection"] == replayed_selection
assert built["pool_rows"] == len(pool) and built["prepared_rows"] == len(prepared.rows)
assert built["exclusions"] == prepared.exclusions
print(
    "OK: split.build(raw) returns the same test rows, training rows, selection, pool and exclusions as the replay."
)
print(
    f"OK: test drawn first; the pool of {len(pool):,} rows excludes all {test['group'].nunique()} test groups "
    f"({len(prepared.rows) - len(pool)} rows removed); selection and training rows reproduce the manifest."
)

ranking = pd.DataFrame({"mutual_information": selection["mutual_information"]})
ranking["decision"] = [
    f"selected #{FROZEN_FEATURES.index(n) + 1}"
    if n in FROZEN_FEATURES
    else "skipped: " + selection["skipped_as_redundant"][n]
    if n in selection["skipped_as_redundant"]
    else "not reached (four already selected)"
    for n in ranking.index
]
ranking

OK: split.build(raw) returns the same test rows, training rows, selection, pool and exclusions as the replay.
OK: test drawn first; the pool of 234,734 rows excludes all 12 test groups (28 rows removed); selection and training rows reproduce the manifest.


,mutual_information,decision
host_entropy,0.121376,selected #1
host_label_count,0.115390,selected #2
host_length,0.113801,skipped: |Spearman| >= 0.7 with host_entropy
host_digit_ratio,0.110528,selected #3
host_hyphen_count,0.059545,selected #4
longest_label_length,0.029827,not reached (four already selected)
tld_length,0.023644,not reached (four already selected)
host_is_ip,0.002293,not reached (four already selected)


## Interpretation

- The snapshot used here is byte-identical to the documented acquisition, and the preparation
  script regenerates the versioned split manifest (every key except the recorded environment)
  with the same seed and rules.
  Anyone with the same UCI files gets the same 36 rows.
- The shortcut audit explains the feature design: legitimate rows are all bare
  `https://www.<host>` homepages, so URL-level and page-level fields are excluded and the four
  frozen features (`host_entropy`, `host_label_count`, `host_digit_ratio`,
  `host_hyphen_count`) come from the host with `www.` removed. Host features still carry
  collection bias, for example phishing pages on free hosting subdomains; they are not
  evidence about current attacks.
- The split is domain-disjoint: no registrable domain appears in both partitions, and the
  feature selection never saw the test groups.
- Limitations: the benchmark is one split from one seed, with 12 test examples, so each test
  example is worth 8.3 points of accuracy. The balanced sample says nothing about production
  prevalence. The seed and sizes were fixed before any model was trained and must not be
  changed after seeing results.

Run next: notebook `02` (feature scaling and training).